# YOLO26n Training

This notebook trains and evaluates YOLO26n on the robotic block dataset.

The model is trained using the fixed training and validation sets created in
`dataset_split.ipynb`.

The final model is evaluated once on the fixed test set.

The main evaluation metrics are:

- Precision
- Recall
- mAP@0.50
- mAP@0.75
- mAP@0.50:0.95
- Per-class AP

Evaluation plots such as the confusion matrix, precision-recall curve and
F1-confidence curve are also drawn.

## 1. Import Libraries

The required libraries are imported below.

PyTorch is used by the YOLO model for deep learning and GPU computation.


Ultralytics provides the YOLO26 implementation used for training and evaluation.

In [2]:
import os
from pathlib import Path
import torch
import ultralytics
from ultralytics import YOLO

## 2. Check the Training Environment

Training will be carried out using my RTX 4060.

This cell checks the installed Ultralytics and PyTorch versions and confirms that
CUDA is available before training begins.

In [3]:
# Print the main software versions used for the experiment.
print(f"Ultralytics version: {ultralytics.__version__}")
print(f"PyTorch version:     {torch.__version__}")

# Check whether PyTorch can access the NVIDIA GPU.
cuda_available = torch.cuda.is_available()

print(f"CUDA available:      {cuda_available}")

# Stop here if CUDA is not available.
if not cuda_available:
    raise RuntimeError(
        "CUDA is not available. "
        "Check the PyTorch and NVIDIA setup before training."
    )

# Print the GPU that will be used for training.
print(
    f"GPU: "
    f"{torch.cuda.get_device_name(0)}"
)

Ultralytics version: 8.4.116
PyTorch version:     2.11.0+cu128
CUDA available:      True
GPU: NVIDIA GeForce RTX 4060


## 3. Set Project and Dataset Paths

The project root is detected automatically.

The split dataset created in the previous notebook is stored in `dataset_split`.

YOLO training outputs are stored under `runs/yolo`.

In [4]:
# Start with the current working directory.
PROJECT_ROOT = Path.cwd().resolve()

# If the notebook is being run from the notebooks folder,
# move up one level to the main project directory.
if not (PROJECT_ROOT / "dataset_split").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

# Location of the fixed train, validation and test dataset.
DATASET_ROOT = PROJECT_ROOT / "dataset_split"

# YOLO requires a YAML file that describes the dataset.
DATA_YAML = DATASET_ROOT / "data.yaml"

# Location of the pretrained YOLO26n starting model.
PRETRAINED_MODEL_PATH = (
        PROJECT_ROOT
        / "models"
        / "pretrained"
        / "yolo26n.pt"
)

# Store YOLO training and evaluation results here.
YOLO_RUNS = PROJECT_ROOT / "runs" / "yolo"

print(f"Project root:     {PROJECT_ROOT}")
print(f"Dataset root:     {DATASET_ROOT}")
print(f"Pretrained model: {PRETRAINED_MODEL_PATH}")
print(f"YOLO runs:        {YOLO_RUNS}")

Project root:     C:\Users\wjohn\Desktop\dobot-thesis
Dataset root:     C:\Users\wjohn\Desktop\dobot-thesis\dataset_split
Pretrained model: C:\Users\wjohn\Desktop\dobot-thesis\models\pretrained\yolo26n.pt
YOLO runs:        C:\Users\wjohn\Desktop\dobot-thesis\runs\yolo


## 4. Check the Dataset

Before training starts, check that all required image and label directories exist.

This helps catch missing or incorrectly created dataset folders before starting a
long training run.

In [5]:
# These folders should have been created by
# 01_dataset_split.ipynb.
required_paths = [
    DATASET_ROOT / "images" / "train",
    DATASET_ROOT / "images" / "val",
    DATASET_ROOT / "images" / "test",
    DATASET_ROOT / "labels" / "train",
    DATASET_ROOT / "labels" / "val",
    DATASET_ROOT / "labels" / "test",
]

# Check each required folder.
for path in required_paths:
    if not path.exists():
        raise FileNotFoundError(
            f"Missing dataset directory: {path}"
        )

print("All required dataset folders were found.")

All required dataset folders were found.


## 5. Create the YOLO Dataset Configuration

Ultralytics YOLO uses a YAML file to locate the training, validation and test images
and to define the object classes.

The four classes are:

- 0: red
- 1: blue
- 2: yellow
- 3: green

These class IDs must match the IDs used in the YOLO annotation files.

In [6]:
# Create the dataset configuration expected by Ultralytics YOLO.
data_yaml_text = f"""path: {DATASET_ROOT.as_posix()}
train: images/train
val: images/val
test: images/test

names:
  0: red
  1: blue
  2: yellow
  3: green
"""

# Save the configuration inside dataset_split.
DATA_YAML.write_text(
    data_yaml_text,
    encoding="utf-8",
)

# Print it so the configuration can be checked before training.
print(DATA_YAML.read_text())

path: C:/Users/wjohn/Desktop/dobot-thesis/dataset_split
train: images/train
val: images/val
test: images/test

names:
  0: red
  1: blue
  2: yellow
  3: green



## 6. Load the Pretrained YOLO26n Model

YOLO26n is the nano sized YOLO26 object detector.

A pretrained model is used as the starting point rather than training the network
from random weights.

The model will then be fine-tuned using the coloured block training dataset.

In [7]:
# Make sure the pretrained YOLO26n model exists
# in the expected project location.
if not PRETRAINED_MODEL_PATH.exists():
    raise FileNotFoundError(
        f"Pretrained model not found: "
        f"{PRETRAINED_MODEL_PATH}"
    )

# Load the pretrained YOLO26n detection model
# from the fixed project path.
model = YOLO(
    str(PRETRAINED_MODEL_PATH)
)

print(
    f"YOLO26n model loaded from: "
    f"{PRETRAINED_MODEL_PATH}"
)

YOLO26n model loaded from: C:\Users\wjohn\Desktop\dobot-thesis\models\pretrained\yolo26n.pt


## 7. Train YOLO26n

The model is trained using the fixed training set.

The validation set is evaluated during training and is used to determine the best
model checkpoint.

The main training settings are:

- 100 maximum epochs
- 640 pixel input size
- automatic batch-size selection
- NVIDIA GPU
- fixed random seed of 42

The test set is not used in this stage.

In [8]:
# Ultralytics performs an internal AMP check using
# the filename "yolo26n.pt".
#
# Temporarily run training from the pretrained-model
# directory so that check finds the existing model
# instead of downloading another copy into Notebooks.
original_working_directory = Path.cwd()

os.chdir(
    PRETRAINED_MODEL_PATH.parent
)

try:
    training_results = model.train(
        data=str(DATA_YAML),

        # Give the model enough time to converge.
        # Early stopping prevents unnecessary training.
        epochs=100,

        # Stop if validation performance has not
        # improved for 20 epochs.
        patience=20,

        # Standard YOLO26 training resolution.
        imgsz=640,

        # Automatically choose a batch size based
        # on the available GPU memory.
        batch=-1,

        # Use the RTX 4060.
        device=0,

        # Keeps Jupyter data loading simple on Windows.
        workers=0,

        # Fixed seed for reproducibility.
        seed=42,

        # Make repeated training as reproducible
        # as possible.
        deterministic=True,

        # Save training results here.
        project=str(YOLO_RUNS),

        # Name of this training run.
        name="yolo26n_blocks",
    )

finally:
    # Return the notebook to its original
    # working directory after training.
    os.chdir(
        original_working_directory
    )

New https://pypi.org/project/ultralytics/8.4.117 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.116  Python-3.14.4 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4060, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\wjohn\Desktop\dobot-thesis\dataset_split\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_

## 8. Load the Best Model

Ultralytics saves model checkpoints during training.

`best.pt` represents the checkpoint that achieved the best validation performance
during training.

This checkpoint is used for the final test-set evaluation instead of simply using
the weights from the final training epoch.

In [9]:
# Use the best checkpoint from the training run
# that was just completed.
BEST_MODEL_PATH = Path(
    model.trainer.best
)

# Make sure the checkpoint exists.
if not BEST_MODEL_PATH.exists():
    raise FileNotFoundError(
        f"Best model not found: {BEST_MODEL_PATH}"
    )

print(f"Best model: {BEST_MODEL_PATH}")

# Load the best checkpoint for final evaluation.
best_model = YOLO(
    str(BEST_MODEL_PATH)
)

Best model: C:\Users\wjohn\Desktop\dobot-thesis\runs\yolo\yolo26n_blocks\weights\best.pt


# Final Test Evaluation

The best YOLO26n checkpoint is now evaluated on the fixed test set.

The test set was not used for model training or model selection.

This provides the final offline detection results that will later be compared with
the Faster R-CNN and OpenCV detector results.

In [10]:
# Evaluate the best trained model on the fixed test set.
test_metrics = best_model.val(
    # Use the same dataset configuration.
    data=str(DATA_YAML),

    # Evaluate the test images rather than the validation images.
    split="test",

    # Use the same image size used during training.
    imgsz=640,

    # Run evaluation on the GPU.
    device=0,

    # Keep Jupyter data loading simple on Windows.
    workers=0,

    # Generate and save evaluation plots.
    plots=True,

    # Store the final test results separately
    # from the training results.
    project=str(YOLO_RUNS),
    name="yolo26n_test",
)

Ultralytics 8.4.116  Python-3.14.4 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4060, 8188MiB)
YOLO26n summary (fused): 122 layers, 2,375,616 parameters, 0 gradients, 5.3 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 2659.6340.5 MB/s, size: 426.2 KB)
val: Scanning C:\Users\wjohn\Desktop\dobot-thesis\dataset_split\labels\test\01_single_blocks\blue_cube.cache... 37 images, 2 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 37/37 15.5Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 3/3 3.2it/s 1.0s0.8s
                   all         37         58      0.963      0.986      0.994      0.993
                   red         16         16      0.956          1      0.995      0.995
                  blue         14         14      0.931      0.971       0.99      0.989
                yellow         14         14      0.965          1      0.995      0.995
                 green         14         14          1      0.973

## 10. Overall Test Metrics

The main overall detection metrics are extracted below.

**Precision** measures how many predicted detections were correct.

**Recall** measures how many ground-truth objects were successfully detected.

**mAP@0.50** measures mean average precision using an IoU threshold of 0.50.

**mAP@0.75** uses the stricter IoU threshold of 0.75 and therefore places more
importance on accurate bounding-box localisation.

**mAP@0.50:0.95** averages AP across IoU thresholds from 0.50 to 0.95 and provides
the main overall object-detection performance measure.

In [11]:
# Extract the overall test metrics.
precision = test_metrics.box.mp
recall = test_metrics.box.mr
map50 = test_metrics.box.map50
map75 = test_metrics.box.map75
map50_95 = test_metrics.box.map

# Print the results clearly.
print("YOLO26n TEST RESULTS")

print(
    f"Precision:     "
    f"{precision:.4f}"
)

print(
    f"Recall:        "
    f"{recall:.4f}"
)

print(
    f"mAP@0.50:      "
    f"{map50:.4f}"
)

print(
    f"mAP@0.75:      "
    f"{map75:.4f}"
)

print(
    f"mAP@0.50:0.95: "
    f"{map50_95:.4f}"
)

YOLO26n TEST RESULTS
Precision:     0.9631
Recall:        0.9858
mAP@0.50:      0.9938
mAP@0.75:      0.9938
mAP@0.50:0.95: 0.9934


## 11. Per-Class Test Results

Overall metrics can hide differences between individual block colours.

The next cell therefore reports performance separately for red, blue, yellow and
green blocks.

This will help determine whether a model has difficulty detecting a particular
colour.

In [12]:
# The class names are kept in the same order
# as the dataset configuration.
class_names = [
    "red",
    "blue",
    "yellow",
    "green",
]

print("PER-CLASS TEST RESULTS")

# ap_class_index tells us which actual dataset class
# each result belongs to.
for result_index, class_id in enumerate(
        test_metrics.ap_class_index
):
    # Get precision, recall, AP50 and AP50-95
    # for this individual class.
    (
        class_precision,
        class_recall,
        class_map50,
        class_map50_95,
    ) = test_metrics.class_result(
        result_index
    )

    # Convert the numeric class ID to its colour name.
    class_name = class_names[
        int(class_id)
    ]

    print()
    print(class_name.upper())

    print(
        f"Precision:     "
        f"{class_precision:.4f}"
    )

    print(
        f"Recall:        "
        f"{class_recall:.4f}"
    )

    print(
        f"mAP@0.50:      "
        f"{class_map50:.4f}"
    )

    print(
        f"mAP@0.50:0.95: "
        f"{class_map50_95:.4f}"
    )

PER-CLASS TEST RESULTS

RED
Precision:     0.9559
Recall:        1.0000
mAP@0.50:      0.9950
mAP@0.50:0.95: 0.9950

BLUE
Precision:     0.9314
Recall:        0.9705
mAP@0.50:      0.9903
mAP@0.50:0.95: 0.9886

YELLOW
Precision:     0.9654
Recall:        1.0000
mAP@0.50:      0.9950
mAP@0.50:0.95: 0.9950

GREEN
Precision:     1.0000
Recall:        0.9728
mAP@0.50:      0.9950
mAP@0.50:0.95: 0.9950


## 12. Evaluation Plots

Ultralytics automatically saves evaluation plots when `plots=True` is used.

These plots provide additional information about the detector and can be used in
the dissertation where useful.

Important outputs include:

- Confusion matrix
- Precision-recall curve
- F1-confidence curve
- Precision-confidence curve
- Recall-confidence curve

The plots are stored in the YOLO test results directory.

## 13. Show the Saved Output Locations

The final cell prints the locations of the training and test outputs.

These directories contain the trained model weights, metric plots and other
Ultralytics results that should be kept for the dissertation analysis.

In [13]:
# Location containing the training results and model weights.
training_output = (
        YOLO_RUNS
        / "yolo26n_blocks"
)

# Location containing the final test evaluation.
test_output = (
        YOLO_RUNS
        / "yolo26n_test"
)

print("Training outputs:")
print(training_output)

print()

print("Test evaluation outputs:")
print(test_output)

Training outputs:
C:\Users\wjohn\Desktop\dobot-thesis\runs\yolo\yolo26n_blocks

Test evaluation outputs:
C:\Users\wjohn\Desktop\dobot-thesis\runs\yolo\yolo26n_test
